In [132]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

In [180]:
N_users = 10000
N_times_train = 10
N_times_val = 5
N_times_prod = 5

# Put an average of 2 users in each category. Not perfect memorization.
N_categories = N_users // 2
# Bounds for the default probabilities
p_min = 0.01
p_max = 0.20

seed = 1235

In [181]:
# Transformations of parameters, do not change these

# Convert min and max probabilities into logits
z_max = np.log(p_max/(1-p_max))
z_min = np.log(p_min/(1-p_min))

In [182]:
# Generate the dataset
rng = np.random.default_rng(seed)

dataset = []  # (t, user_idx, x0, x1, y)

for idx_user in range(N_users):
    cat_user = rng.choice(N_categories)
    # Run a simulation forward in time to determine when the user defaults (if ever)
    for t in range(N_times_train + N_times_val + N_times_prod):
        # Reset user circumstances each time
        z_user = rng.uniform(low=z_min, high=z_max)
        p_user = 1 / (1 + np.exp(-z_user))
        
        does_user_default = rng.binomial(p=p_user, n=1)
        dataset.append((t, idx_user, z_user, cat_user, does_user_default))
        if does_user_default:
            break

# Make it a pandas DataFrame for easier manipulation
dataset_pd = pd.DataFrame(
    dataset,
    columns=['t', 'user_idx', 'x0', 'x1', 'y']
)

In [183]:
# Fit a simple model to the training data and show that the regression coefficient is 1.0
lr_simple_model = LogisticRegression()
lr_simple_model.fit(
    X = dataset_pd[dataset_pd['t'] < N_times_train][['x0']],
    y = dataset_pd[dataset_pd['t'] < N_times_train]['y'],
)

print(f'Regression coefficient of simple model: {lr_simple_model.coef_[0, 0]}')

Regression coefficient of simple model: 0.9878547167825326


In [184]:
def run_experiment(
    dataset_pd: pd.DataFrame,
    battenburg_validation: bool,
):
    # This section gets affected by our choice of val scheme
    df_train = dataset_pd[dataset_pd['t'] < N_times_train]
    df_val = dataset_pd[(dataset_pd['t'] >= N_times_train) & (dataset_pd['t'] < N_times_train + N_times_val)]
    
    if battenburg_validation:
        # Randomly assign users to train or val
        users_train_idx = rng.choice(a=N_users, size=N_users//2, replace=False)
        df_train = df_train[df_train['user_idx'].isin(users_train_idx)]
        df_val = df_val[~df_val['user_idx'].isin(users_train_idx)]

        print(sorted(list(set(df_train['user_idx'])))[:10])
        print(sorted(list(set(df_val['user_idx'])))[:10])
    
    # Everything below here does not
    X_train = df_train.drop(columns=['t', 'user_idx', 'y'])
    y_train = df_train['y']
    
    X_val = df_val.drop(columns=['t', 'user_idx', 'y'])
    y_val = df_val['y']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('none', 'passthrough', ['x0']),
            ('one-hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['x1'])
        ]
    )
    
    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('classifier', LogisticRegression())
        ]
    )
    
    pipeline.fit(X_train, y_train)
    
    roc_auc_train = roc_auc_score(y_true=y_train, y_score=pipeline.predict_proba(X_train)[:, 1])
    pr_auc_train = average_precision_score(y_true=y_train, y_score=pipeline.predict_proba(X_train)[:, 1])
    log_loss_train = log_loss(y_true=y_train, y_pred=pipeline.predict_proba(X_train)[:, 1])
    
    roc_auc_val = roc_auc_score(y_true=y_val, y_score=pipeline.predict_proba(X_val)[:, 1])
    pr_auc_val = average_precision_score(y_true=y_val, y_score=pipeline.predict_proba(X_val)[:, 1])
    log_loss_val = log_loss(y_true=y_val, y_pred=pipeline.predict_proba(X_val)[:, 1])
    
    # Retrain for prod
    df_retrain = dataset_pd[dataset_pd['t'] < N_times_train + N_times_val]
    df_prod = dataset_pd[dataset_pd['t'] >= N_times_train + N_times_val]

    X_retrain = df_retrain.drop(columns=['t', 'user_idx', 'y'])
    y_retrain = df_retrain['y']
    
    X_prod = df_prod.drop(columns=['t', 'user_idx', 'y'])
    y_prod = df_prod['y']
    
    pipeline.fit(X_retrain, y_retrain)
    roc_auc_retrain = roc_auc_score(y_true=y_retrain, y_score=pipeline.predict_proba(X_retrain)[:, 1])
    pr_auc_retrain = average_precision_score(y_true=y_retrain, y_score=pipeline.predict_proba(X_retrain)[:, 1])
    log_loss_retrain = log_loss(y_true=y_retrain, y_pred=pipeline.predict_proba(X_retrain)[:, 1])
    roc_auc_prod = roc_auc_score(y_true=y_prod, y_score=pipeline.predict_proba(X_prod)[:, 1])
    pr_auc_prod = average_precision_score(y_true=y_prod, y_score=pipeline.predict_proba(X_prod)[:, 1])
    log_loss_prod = log_loss(y_true=y_prod, y_pred=pipeline.predict_proba(X_prod)[:, 1])

    avg_p_for_pos_train = pipeline.predict_proba(X_train)[:, 1] @ y_train / np.sum(y_train)
    avg_p_for_pos_val = pipeline.predict_proba(X_val)[:, 1] @ y_val / np.sum(y_val)
    avg_p_for_pos_retrain = pipeline.predict_proba(X_retrain)[:, 1] @ y_retrain / np.sum(y_retrain)
    avg_p_for_pos_prod = pipeline.predict_proba(X_prod)[:, 1] @ y_prod / np.sum(y_prod)
    
    return (
        roc_auc_train,
        roc_auc_val,
        roc_auc_retrain,
        roc_auc_prod,
        pr_auc_train,
        pr_auc_val,
        pr_auc_retrain,
        pr_auc_prod,
        log_loss_train,
        log_loss_val,
        log_loss_retrain,
        log_loss_prod,
        avg_p_for_pos_train,
        avg_p_for_pos_train,
        avg_p_for_pos_retrain,
        avg_p_for_pos_train,
    )

In [189]:
(pipeline.predict_proba(X_train)[:, 1] @ y_train) / np.sum(y_train)

np.float64(0.12482569655833167)

In [185]:
run_experiment(dataset_pd, battenburg_validation=False)

(np.float64(0.8166967479755581),
 np.float64(0.7232486823191184),
 np.float64(0.8051451535305774),
 np.float64(0.7097597047487433),
 np.float64(0.27552886108714497),
 np.float64(0.13336164448765514),
 np.float64(0.2527094100615979),
 np.float64(0.1358803484203939),
 0.19910018452046677,
 0.22741643531652345,
 0.20272742574205255,
 0.23639760954980987)

In [186]:
run_experiment(dataset_pd, battenburg_validation=True)

[5, 6, 8, 11, 14, 19, 20, 22, 23, 26]
[0, 2, 7, 16, 17, 18, 21, 24, 25, 29]


(np.float64(0.840350000627772),
 np.float64(0.7311243303799959),
 np.float64(0.8051451535305774),
 np.float64(0.7097597047487433),
 np.float64(0.3350135079475324),
 np.float64(0.1363717753500373),
 np.float64(0.2527094100615979),
 np.float64(0.1358803484203939),
 0.19454449537620877,
 0.22118679185891657,
 0.20272742574205255,
 0.23639760954980987)

In [187]:
dataset_pd

,t,user_idx,x0,x1,y
0,0,0,-4.006474,1878,0
1,1,0,-1.905789,1878,0
2,2,0,-3.654247,1878,0
3,3,0,-2.691103,1878,0
4,4,0,-2.510687,1878,0
...,...,...,...,...,...
113162,7,9999,-3.830854,2958,0
113163,8,9999,-2.462617,2958,0
113164,9,9999,-2.483117,2958,0
113165,10,9999,-4.095047,2958,0


In [179]:
pipeline.steps[1][1].coef_

array([[ 1.07672272,  0.47831347, -0.40957374, ..., -0.478598  ,
        -0.09502617,  0.06142921]])